# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 291.06it/s]


2026-09-01 12:45:02.702 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:903 - Data batch-empirical estimation of propensity score.


2026-09-01 12:45:02.709 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:952 - Data prediction of expected reward based on gbm model.


2026-09-01 12:45:04.103 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1069 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

2026-09-01 12:45:37.003 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-09-01 12:45:37.304 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-09-01 12:45:37.605 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.504833,0.471132,0.537309,0.016728,b-ipw,reward_0
1,0.501351,0.500421,0.502303,0.000481,dm,reward_0
2,0.501330,0.469432,0.534707,0.016463,dr,reward_0
3,0.501351,0.500397,0.502269,0.000474,dros-opt,reward_0
4,0.501330,0.469701,0.534241,0.016433,dros-pess,reward_0
5,0.501354,0.468232,0.534576,0.016878,ipw,reward_0
6,0.500810,0.468967,0.535271,0.016764,rep,reward_0
7,0.501330,0.469840,0.534749,0.016358,sndr,reward_0
8,0.501288,0.469611,0.533923,0.016404,snips,reward_0
9,0.501330,0.469352,0.532498,0.016232,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 322.59it/s]


2026-09-01 12:45:38.057 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1301 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<09:14,  1.80it/s]

...

2026-09-01 12:46:32.628 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-09-01 12:46:32.933 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-09-01 12:46:33.237 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.504918,0.473769,0.536490,0.016035,b-ipw,reward_0
1,0.501379,0.500435,0.502313,0.000480,dm,reward_0
2,0.503733,0.472082,0.535400,0.016027,dr,reward_0
3,0.501379,0.500426,0.502282,0.000475,dros-opt,reward_0
4,0.503733,0.472122,0.534450,0.016029,dros-pess,reward_0
5,0.504081,0.472584,0.534905,0.015945,ipw,reward_0
6,0.503283,0.471073,0.534724,0.016089,rep,reward_0
7,0.503732,0.471937,0.534019,0.015831,sndr,reward_0
8,0.503757,0.473040,0.535229,0.015865,snips,reward_0
9,0.503733,0.473001,0.534303,0.015884,sg-dr,reward_0
